# Gold_genre_performance

## Import and Load

In [38]:
import pandas as pd

from ecommerce_ingestion.config.paths import DB_OUTPUT_GOLD
from ecommerce_ingestion.utils.genre_unifier_service import get_genre_unified_genre_data


In [39]:
gold_genre_performance_raw = pd.read_parquet(
    DB_OUTPUT_GOLD / "gold_game_catalog.parquet")
gold_genre_performance_raw.shape

(2976, 18)

In [40]:

gold_genre_performance_filtered = get_genre_unified_genre_data(
    gold_genre_performance_raw)

col_list = ["game_id", 
            "user_score", 
            "meta_score", 
            "current_price", 
            "stock_status", 
            "unified_genre"]

gold_genre_performance = gold_genre_performance_filtered[col_list]
gold_genre_performance.shape

(4661, 6)

## Grouping

Columns

número de juegos
precio medio
meta score medio
user score medio
porcentaje en stock
plataformas donde aparece
índice de atractivo del género

In [41]:
gold_genre_performance = gold_genre_performance.groupby("unified_genre").agg(
     game_count=("game_id", "count"),
     average_user_score=("user_score", "mean"),
     average_meta_score=("meta_score", "mean"),
     average_price=("current_price", "mean"),
     in_stock_percentage=("stock_status", lambda x: (x == "in_stock").mean() * 100)
 )
gold_genre_performance["score_price"] = gold_genre_performance[
    "average_user_score"] / gold_genre_performance["average_price"]
gold_genre_performance.shape

(17, 6)

In [42]:
gold_genre_performance.head()

,game_count,average_user_score,average_meta_score,average_price,in_stock_percentage,score_price
unified_genre,,,,,,
action,1294,74.940495,77.194745,75.895719,51.545595,0.987414
adventure,268,71.600746,75.750000,72.426567,48.880597,0.988598
arcade,255,74.521569,77.431373,75.503725,51.372549,0.986992
board_game,50,68.900000,78.780000,69.890000,46.000000,0.985835
fighting,97,76.762887,77.814433,77.763196,55.670103,0.987136


In [43]:
gold_genre_performance.isna().sum()

game_count             0
average_user_score     0
average_meta_score     0
average_price          0
in_stock_percentage    0
score_price            0
dtype: int64

## Save Data

In [44]:
gold_genre_performance.to_parquet(DB_OUTPUT_GOLD / "gold_genre_performance.parquet")